# Kumpulan Script Python (Tugas B3 ABSA Using BERT)

Notebook ini adalah gabungan dari seluruh script Python yang dikerjakan secara berurutan sesuai dengan tahapan tugas.

In [ ]:
# Sel ini ditambahkan khusus untuk eksekusi lokal di folder notebooks/tugas_b3/
# Ini akan menggeser working directory kembali ke root agar pemanggilan folder 'data/' tidak error.
import os
import sys
if os.path.basename(os.getcwd()) == 'tugas_b3':
    os.chdir('../../')
    print('Working directory digeser ke:', os.getcwd())

## Tahap 1: preprocess_dataset.py

In [ ]:
import pandas as pd
import re

# Load dataset hasil tahap 2
df = pd.read_csv('data/processed/dataset_absa_labeled.csv')

# Kamus Normalisasi Kata Gaul/Typo (Bisa ditambah sesuai kebutuhan)
norm_dict = {
    'emg': 'memang',
    'bgtttt': 'banget',
    'bgt': 'banget',
    'tp': 'tapi',
    'dpt': 'dapat',
    'yg': 'yang',
    'dg': 'dengan',
    'utk': 'untuk'
}

def clean_text(text):
    text = str(text)
    # 1. Case Folding
    text = text.lower()
    
    # 2. Penghapusan Karakter Khusus (Hapus semua kecuali huruf dan angka)
    # Kita biarkan spasi, huruf, dan angka.
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Menghapus extra spasi
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 3. Tokenisasi (Pecah berdasarkan spasi)
    tokens = text.split()
    
    # 4. Normalisasi Kata
    normalized_tokens = [norm_dict.get(token, token) for token in tokens]
    
    # Catatan: Kita tidak melakukan Penghapusan Stopword karena untuk BERT, 
    # stopword (seperti 'dan', 'yang', 'tidak') sangat penting untuk memahami konteks kalimat.
    
    # Gabungkan kembali menjadi kalimat bersih untuk input model
    clean_sentence = ' '.join(normalized_tokens)
    return clean_sentence

# Terapkan fungsi cleaning ke kolom Review
df['Review_Clean'] = df['Review'].apply(clean_text)

# Simpan dataset yang sudah di-preprocess
df.to_csv('data/processed/dataset_absa_preprocessed.csv', index=False)

print("Dataset berhasil di-preprocess (Case Folding, Penghapusan Karakter, Normalisasi).")
print("Hasil disimpan di: data/processed/dataset_absa_preprocessed.csv")
print("\nContoh Hasil:")
print("Asli   :", df['Review'].iloc[0])
print("Bersih :", df['Review_Clean'].iloc[0])



Mengeksekusi cell 3...
Dataset berhasil di-preprocess (Case Folding, Penghapusan Karakter, Normalisasi).
Hasil disimpan di: data/processed/dataset_absa_preprocessed.csv
Contoh Hasil:
Asli   : Di sisi lain, sambung dia, Pemerintah Provinsi (Pemprov) DKI Jakarta juga harus jelas memetakan jalur akademik para penerima LPDP, sehingga ketika mereka sudah lulus, dapat berpartisipasi membangun Jakarta melalui sektor yang tepat.
Bersih : di sisi lain sambung dia pemerintah provinsi pemprov dki jakarta juga harus jelas memetakan jalur akademik para penerima lpdp sehingga ketika mereka sudah lulus dapat berpartisipasi membangun jakarta melalui sektor yang tepat


## Tahap 2: format_bert_input.py

In [ ]:
import pandas as pd

# Load dataset hasil preprocessing
df = pd.read_csv('data/processed/dataset_absa_preprocessed.csv')

def create_bert_input(aspect, review):
    # Menyisipkan [CLS] di awal, [SEP] di antara aspek dan review, dan [SEP] di akhir.
    return f"[CLS] {str(aspect)} [SEP] {str(review)} [SEP]"

# Buat kolom baru 'BERT_Input'
df['BERT_Input'] = df.apply(lambda row: create_bert_input(row['Aspect'], row['Review_Clean']), axis=1)

# Simpan hasilnya
df.to_csv('data/processed/dataset_absa_bert_ready.csv', index=False)

print("Berhasil membentuk format input BERT!")
print("Contoh struktur baris pertama:")
print(df['BERT_Input'].iloc[0])



Mengeksekusi cell 5...
Berhasil membentuk format input BERT!
Contoh struktur baris pertama:
[CLS] kualitas pendidikan [SEP] di sisi lain sambung dia pemerintah provinsi pemprov dki jakarta juga harus jelas memetakan jalur akademik para penerima lpdp sehingga ketika mereka sudah lulus dapat berpartisipasi membangun jakarta melalui sektor yang tepat [SEP]


## Tahap 3: bert_tokenize.py

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Kita pakai model pre-trained BERT berbahasa Indonesia (IndoBERT)
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

# Load dataset
df = pd.read_csv('data/processed/dataset_absa_preprocessed.csv')

# Ambil satu contoh (baris pertama)
aspect = str(df['Aspect'].iloc[0])
review = str(df['Review_Clean'].iloc[0])

print("==== TEKS MENTAH ====")
print(f"Aspect : {aspect}")
print(f"Review : {review}\n")

# Tokenisasi pasangan kalimat (ABSA)
# Tokenizer BERT otomatis menyisipkan [CLS] dan [SEP] jika kita beri dua input
encoded = tokenizer(
    text=aspect, 
    text_pair=review, 
    padding='max_length', 
    max_length=64, # Kita batasi panjang arraynya agar output terminal tidak terlalu penuh
    truncation=True
)

print("==== HASIL REPRESENTASI NUMERIK DARI TOKENIZER ====\n")
print(f"1. input_ids      :\n{encoded['input_ids']}\n")
print(f"2. attention_mask :\n{encoded['attention_mask']}\n")
print(f"3. token_type_ids :\n{encoded['token_type_ids']}\n")

# Mari kita lihat bagaimana angka-angka input_ids diterjemahkan kembali ke kata-kata (untuk bukti adanya [CLS] dan [SEP])
print("==== DECODE KEMBALI INPUT_IDS ====")
print(tokenizer.decode(encoded['input_ids']))



Mengeksekusi cell 7...
==== TEKS MENTAH ====
Aspect : kualitas pendidikan
Review : di sisi lain sambung dia pemerintah provinsi pemprov dki jakarta juga harus jelas memetakan jalur akademik para penerima lpdp sehingga ketika mereka sudah lulus dapat berpartisipasi membangun jakarta melalui sektor yang tepat
==== HASIL REPRESENTASI NUMERIK DARI TOKENIZER ====
1. input_ids      :
[2, 1553, 701, 3, 26, 2123, 245, 10691, 364, 877, 2142, 9648, 4005, 678, 186, 308, 1127, 19582, 8403, 5, 2976, 6489, 383, 6325, 7354, 14127, 485, 640, 267, 259, 4308, 173, 7344, 2255, 678, 709, 3664, 34, 1234, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2. attention_mask :
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
3. token_type_ids :
[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

## Tahap 4: bert_classifier_model.py

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel

class ABSABertClassifier(nn.Module):
    """
    Simulasi Arsitektur Model untuk Soal 6
    Menunjukkan alur input dari BERT hingga keluar menjadi skor Prediksi Sentimen
    """
    def __init__(self, num_classes=3):
        super(ABSABertClassifier, self).__init__()
        
        # 1. Lapisan BERT Pre-trained (Sebagai Feature Extractor / Encoder)
        self.bert = BertModel.from_pretrained('indobenchmark/indobert-base-p1')
        
        # 2. Lapisan Dropout (Untuk mencegah overfitting selama Fine-Tuning)
        self.dropout = nn.Dropout(p=0.3)
        
        # 3. Lapisan Linear Classifier (Classification Head)
        # BERT base memiliki hidden size sebesar 768 dimensi
        # Dikerucutkan menjadi 3 dimensi (Positif, Negatif, Netral)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask, token_type_ids):
        # Tahap 1: Input masuk ke Transformer Encoder
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        
        # Tahap 2: Ekstraksi fitur [CLS] (Berada di indeks ke-0 pada last_hidden_state atau melalui pooler_output)
        # pooler_output = Representasi [CLS] yang sudah melewati Linear layer tambahan dan aktivasi Tanh
        pooled_output = outputs.pooler_output 
        
        # Tahap 3: Melewati lapisan Dropout
        pooled_output = self.dropout(pooled_output)
        
        # Tahap 4: Menghasilkan skor akhir (Logits) melalui Layer Klasifikasi Linear
        logits = self.classifier(pooled_output)
        
        # (Fungsi Softmax biasanya digabung dengan Loss Function di PyTorch seperti CrossEntropyLoss)
        return logits




Mengeksekusi cell 9...


## Tahap 5: check_distribution.py

In [ ]:
import pandas as pd

df = pd.read_csv('data/processed/dataset_absa_bert_ready.csv')

print("==== STATISTIK KESELURUHAN ====")
print(f"Total Data: {len(df)} baris")
print("\nDistribusi Sentimen Keseluruhan:")
print(df['Sentiment'].value_counts())

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print("\n==== PEMBAGIAN TRAINING (80%) ====")
print(f"Total Data Train: {len(train_df)} baris")
print("Distribusi Sentimen di Data Latih:")
print(train_df['Sentiment'].value_counts())

print("\n==== PEMBAGIAN TESTING (20%) ====")
print(f"Total Data Test: {len(test_df)} baris")
print("Distribusi Sentimen di Data Uji:")
print(test_df['Sentiment'].value_counts())



Mengeksekusi cell 11...
==== STATISTIK KESELURUHAN ====
Total Data: 100 baris
Distribusi Sentimen Keseluruhan:
Sentiment
Netral     67
Negatif    24
Positif     9
Name: count, dtype: int64
==== PEMBAGIAN TRAINING (80%) ====
Total Data Train: 80 baris
Distribusi Sentimen di Data Latih:
Sentiment
Netral     51
Negatif    23
Positif     6
Name: count, dtype: int64
==== PEMBAGIAN TESTING (20%) ====
Total Data Test: 20 baris
Distribusi Sentimen di Data Uji:
Sentiment
Netral     16
Positif     3
Negatif     1
Name: count, dtype: int64


## Tahap 6: train_absa.py

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import os

# Mematikan peringatan Wandb
os.environ["WANDB_DISABLED"] = "true"

print("1. Membaca Dataset...")
df = pd.read_csv('data/processed/dataset_absa_bert_ready.csv')

# Mapping Sentimen ke angka
label_map = {"Negatif": 0, "Netral": 1, "Positif": 2}
df['Label'] = df['Sentiment'].map(label_map)

# Kita cukup pakai kolom BERT_Input karena struktur [CLS] Aspek [SEP] Review [SEP] sudah disiapkan 
# Tapi karena tokenizer.encode() lebih disarankan menerima pair secara utuh, kita split lagi (atau gunakan format asli)
aspects = df['Aspect'].tolist()
reviews = df['Review_Clean'].tolist()
labels = df['Label'].tolist()

print("2. Melakukan Tokenisasi menggunakan IndoBERT...")
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

# Tokenisasi batch
encodings = tokenizer(
    aspects, 
    reviews, 
    truncation=True, 
    padding='max_length', 
    max_length=64
)

# Pemisahan Train/Test Split Sederhana (80% Train, 20% Eval)
split_idx = int(len(labels) * 0.8)
train_encodings = {key: val[:split_idx] for key, val in encodings.items()}
eval_encodings  = {key: val[split_idx:] for key, val in encodings.items()}
train_labels = labels[:split_idx]
eval_labels = labels[split_idx:]

class ABSADataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ABSADataset(train_encodings, train_labels)
eval_dataset = ABSADataset(eval_encodings, eval_labels)

print("3. Menyiapkan Arsitektur Model IndoBERT Classification...")
# Inisialisasi model pre-trained untuk klasifikasi dengan 3 label
model = AutoModelForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p1", 
    num_labels=3,
    ignore_mismatched_sizes=True
)

# Argumen Pelatihan
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,              # Train 3 epoch
    per_device_train_batch_size=8,   # Ukuran batch training
    per_device_eval_batch_size=8,    # Ukuran batch evaluasi
    learning_rate=2e-5,              # Kecepatan belajar standar BERT
    eval_strategy="epoch",           # Evaluasi dilakukan setiap 1 epoch selesai
    logging_dir='./logs',
    logging_steps=5,
    save_strategy="epoch"
)

# Trainer dari HuggingFace
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

print("4. MEMULAI PROSES FINE-TUNING! Harap tunggu (mungkin butuh beberapa menit di CPU)...")
trainer.train()

print("\nPelatihan Selesai! Model disimpan sementara di folder ./results")



Mengeksekusi cell 13...
1. Membaca Dataset...
2. Melakukan Tokenisasi menggunakan IndoBERT...
3. Menyiapkan Arsitektur Model IndoBERT Classification...
4. MEMULAI PROSES FINE-TUNING! Harap tunggu (mungkin butuh beberapa menit di CPU)...
{'loss': '0.9735', 'grad_norm': '4.604', 'learning_rate': '1.733e-05', 'epoch': '0.5'}
{'loss': '0.737', 'grad_norm': '6.295', 'learning_rate': '1.4e-05', 'epoch': '1'}
{'eval_loss': '0.8329', 'eval_runtime': '0.0542', 'eval_samples_per_second': '368.9', 'eval_steps_per_second': '55.33', 'epoch': '1'}
{'loss': '0.6996', 'grad_norm': '5.104', 'learning_rate': '1.067e-05', 'epoch': '1.5'}
{'loss': '0.7078', 'grad_norm': '10.45', 'learning_rate': '7.333e-06', 'epoch': '2'}
{'eval_loss': '0.8197', 'eval_runtime': '0.0514', 'eval_samples_per_second': '388.8', 'eval_steps_per_second': '58.32', 'epoch': '2'}
{'loss': '0.5068', 'grad_norm': '4.996', 'learning_rate': '4e-06', 'epoch': '2.5'}
{'loss': '0.6788', 'grad_norm': '7.544', 'learning_rate': '6.667e-07', 

## Tahap 7: evaluate_model.py

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import os

# Mematikan peringatan Wandb
os.environ["WANDB_DISABLED"] = "true"

df = pd.read_csv('data/processed/dataset_absa_bert_ready.csv')
label_map = {"Negatif": 0, "Netral": 1, "Positif": 2}
df['Label'] = df['Sentiment'].map(label_map)

aspects = df['Aspect'].tolist()
reviews = df['Review_Clean'].tolist()
labels = df['Label'].tolist()

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
encodings = tokenizer(aspects, reviews, truncation=True, padding='max_length', max_length=64)

split_idx = int(len(labels) * 0.8)
eval_encodings  = {key: val[split_idx:] for key, val in encodings.items()}
eval_labels = labels[split_idx:]

class ABSADataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

eval_dataset = ABSADataset(eval_encodings, eval_labels)

# Mencari checkpoint terakhir di folder results
import glob
checkpoints = glob.glob('./results/checkpoint-*')
if not checkpoints:
    print("Tidak ditemukan checkpoint model. Pastikan training sudah selesai.")

latest_checkpoint = max(checkpoints, key=os.path.getctime)
print(f"Loading model dari: {latest_checkpoint}")

model = AutoModelForSequenceClassification.from_pretrained(latest_checkpoint, num_labels=3, ignore_mismatched_sizes=True)

trainer = Trainer(model=model)
predictions = trainer.predict(eval_dataset)
preds = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support

cm = confusion_matrix(eval_labels, preds, labels=[0, 1, 2])
acc = accuracy_score(eval_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(eval_labels, preds, average='macro', zero_division=0)

print("\n==== HASIL EVALUASI MODEL ====")
print("Confusion Matrix:")
print(cm)
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

# Simpan hasil ini ke file teks sementara agar mudah dibaca script docx
with open('data/processed/eval_metrics.txt', 'w') as f:
    f.write(f"{acc:.4f}\n{precision:.4f}\n{recall:.4f}\n{f1:.4f}\n{cm.tolist()}")



Mengeksekusi cell 15...
Loading model dari: ./results\checkpoint-30
==== HASIL EVALUASI MODEL ====
Confusion Matrix:
[[ 1  0  0]
 [ 0 16  0]
 [ 0  3  0]]
Accuracy : 0.7000
Precision: 0.3542
Recall   : 0.6042
F1-Score : 0.4042


## Tahap 8: predict_custom.py

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import glob
import os

os.environ["WANDB_DISABLED"] = "true"

# 1. Cari checkpoint terakhir
checkpoints = glob.glob('./results/checkpoint-*')
if not checkpoints:
    print("Checkpoint tidak ditemukan.")

latest_checkpoint = max(checkpoints, key=os.path.getctime)

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
model = AutoModelForSequenceClassification.from_pretrained(latest_checkpoint, num_labels=3, ignore_mismatched_sizes=True)

# 2. Kalimat uji (Kasus LPDP Multi-Aspek)
kalimat = "Fasilitas beasiswa LPDP sangat memadai dan membanggakan, namun proses seleksi wawancaranya terasa sangat memberatkan."
aspek_list = ["fasilitas beasiswa", "seleksi wawancara"]

print("=== HASIL PREDIKSI KASUS ABSA ===")
print(f"Kalimat Review: {kalimat}\n")

label_map_rev = {0: "Negatif", 1: "Netral", 2: "Positif"}

results = []
for aspek in aspek_list:
    inputs = tokenizer(aspek, kalimat, return_tensors="pt", truncation=True, padding='max_length', max_length=64)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred_id = np.argmax(logits.numpy(), axis=1)[0]
        pred_label = label_map_rev[pred_id]
        
    print(f"Aspek    : {aspek}")
    print(f"Sentimen : {pred_label}\n")
    results.append(pred_label)

# Simpan hasil sementara untuk dipakai di docx generator
with open('data/processed/predict_custom.txt', 'w') as f:
    f.write(f"{kalimat}\n{aspek_list[0]}|{results[0]}\n{aspek_list[1]}|{results[1]}")



Mengeksekusi cell 17...
=== HASIL PREDIKSI KASUS ABSA ===
Kalimat Review: Fasilitas beasiswa LPDP sangat memadai dan membanggakan, namun proses seleksi wawancaranya terasa sangat memberatkan.
Aspek    : fasilitas beasiswa
Sentimen : Netral
Aspek    : seleksi wawancara
Sentimen : Netral


## Tahap 9: analyze_errors.py

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import glob
import os

os.environ["WANDB_DISABLED"] = "true"

# 1. Load Model Terakhir
checkpoints = glob.glob('./results/checkpoint-*')
latest_checkpoint = max(checkpoints, key=os.path.getctime) if checkpoints else "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
model = AutoModelForSequenceClassification.from_pretrained(latest_checkpoint, num_labels=3, ignore_mismatched_sizes=True)

label_map_rev = {0: "Negatif", 1: "Netral", 2: "Positif"}

# 2. Daftar Kasus Ekstrim
kasus_ekstrim = [
    {
        "jenis": "Negasi",
        "kalimat": "Saya sama sekali tidak membenci kebijakan baru sistem LPDP.",
        "aspek": "kebijakan sistem"
    },
    {
        "jenis": "Sarkasme",
        "kalimat": "Wah hebat sekali pelayanan LPDP, uang sakunya sampai telat cair 3 bulan berturut-turut.",
        "aspek": "pelayanan"
    },
    {
        "jenis": "Kalimat Panjang",
        "kalimat": "Meskipun pada awalnya saya merasa sangat ragu dan bimbang dengan berbagai macam persyaratan rumit yang dituntut oleh pihak penyelenggara beasiswa dari pusat di mana kita harus menyertakan ribuan berkas yang membingungkan sehingga pada akhirnya dana pendidikan tersebut cair juga.",
        "aspek": "dana pendidikan"
    },
    {
        "jenis": "Kata Ambigu",
        "kalimat": "Persaingan proses seleksi LPDP tahun ini benar-benar gila.",
        "aspek": "proses seleksi"
    }
]

print("=== PENGUJIAN ANALISIS KESALAHAN MODEL (SOAL 10) ===\n")

for kasus in kasus_ekstrim:
    inputs = tokenizer(kasus["aspek"], kasus["kalimat"], return_tensors="pt", truncation=True, padding='max_length', max_length=64)
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = np.argmax(outputs.logits.numpy(), axis=1)[0]
        pred_label = label_map_rev[pred_id]
        
    print(f"[{kasus['jenis']}]")
    print(f"Kalimat : {kasus['kalimat']}")
    print(f"Aspek   : {kasus['aspek']}")
    print(f"Tebakan : {pred_label}\n")



Mengeksekusi cell 19...
=== PENGUJIAN ANALISIS KESALAHAN MODEL (SOAL 10) ===
[Negasi]
Kalimat : Saya sama sekali tidak membenci kebijakan baru sistem LPDP.
Aspek   : kebijakan sistem
Tebakan : Negatif
[Sarkasme]
Kalimat : Wah hebat sekali pelayanan LPDP, uang sakunya sampai telat cair 3 bulan berturut-turut.
Aspek   : pelayanan
Tebakan : Positif
[Kalimat Panjang]
Kalimat : Meskipun pada awalnya saya merasa sangat ragu dan bimbang dengan berbagai macam persyaratan rumit yang dituntut oleh pihak penyelenggara beasiswa dari pusat di mana kita harus menyertakan ribuan berkas yang membingungkan sehingga pada akhirnya dana pendidikan tersebut cair juga.
Aspek   : dana pendidikan
Tebakan : Negatif
[Kata Ambigu]
Kalimat : Persaingan proses seleksi LPDP tahun ini benar-benar gila.
Aspek   : proses seleksi
Tebakan : Negatif
